<h1 style="color:magenta;">Entrenamiento de CNN con NASA Score Function en su función de pérdida.</h1>
<p> Valentina Arce España <p>
<p> Febrero 17, 2026 <p>

La NASA (Scoring Function) al ser puramente exponencial, puede causar gradientes inestables si el error es muy grande al inicio. LINEX (Linear-Exponential Loss) es la solución perfecta porque:

1. Para errores pequeños, se comporta casi linealmente (como el MAE/MSE), lo que estabiliza el entrenamiento.

2. Para errores grandes (hacia el lado peligroso), crece exponencialmente.

### Definir la función de pérdida LINEX personalizada
Para RUL, queremos penalizar las predicciones tardías (Overestimation).

* Si $RUL_{real} = 10$ y $RUL_{pred} = 50$ (Error = -40), esto es peligroso (creemos que el motor está bien y se va a caer). Queremos una penalización gigante.
* Si $RUL_{real} = 50$ y $RUL_{pred} = 10$ (Error = +40), esto es "seguro" (mantenimiento temprano). Queremos una penalización suave.

Matemáticamente, para la función LINEX:

$L(e) = e^{a \cdot e} - a \cdot e - 1$

Donde $e = y_{true} - y_{pred}$.

Para penalizar cuando $y_{pred} > y_{true}$ (o sea, $e$ es negativo), necesitamos que a sea negativo.

In [3]:
import tensorflow as tf
# Esto crea una copia totalmente independiente. Si rompes 'model_linex', 'model' sigue vivo.
model_linex = tf.keras.models.load_model('cnn_baseline_rmse_8.13.keras')
print("✅ Copia creada: 'model_linex'. Trabajaremos sobre esta.")

✅ Copia creada: 'model_linex'. Trabajaremos sobre esta.


In [4]:
import tensorflow.keras.backend as K

import tensorflow.keras.backend as K

def linex_loss(y_true, y_pred):
    # 'a' negativo penaliza fuertemente las predicciones tardías (peligrosas)
    a = -0.1 
    error = y_true - y_pred
    
    # Fórmula LINEX: exp(a*e) - a*e - 1
    loss = K.exp(a * error) - (a * error) - 1
    return K.mean(loss)

In [5]:
from tensorflow.keras.optimizers import Adam

# 1. Compilamos SOLO la copia con la nueva Loss
# Usamos un learning rate bajito (0.0001) para no dañar los pesos que ya aprendió
model_linex.compile(optimizer=Adam(learning_rate=0.0001), loss=linex_loss)

print("Entrenando la copia con LINEX...")

# 2. Entrenamos SOLO la copia
# Usamos pocas épocas (ej. 5 o 10) porque es solo un ajuste fino
history_linex = model_linex.fit(
    X_train_w, y_train_w,
    epochs=10, 
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

Entrenando la copia con LINEX...


NameError: name 'X_train_w' is not defined